# Stratified Evaluation of Medication Extraction Systems

**Project:** Clinical Medication Extraction — GPT-4o vs. BioMistral vs. Regex Baseline  
**Datasets:** n2c2 2018 Track 2 (regex, BioMistral), MTSamples (GPT-4o)  
**Author:** Jackie Kim

---

## Purpose

This notebook operationalizes the core research question: **do extraction failures distribute randomly across drug types, or concentrate systematically in specific drug classes?**

Aggregate F1 is the standard benchmark metric for NER tasks. It is also misleading in clinical contexts. A model with Drug F1 = 0.74 overall and Drug F1 = 0.14 on oncology agents is not a 0.74 model for any downstream study that touches chemotherapy cohorts. This notebook makes that argument quantitatively.

### Sections
1. Load and validate results
2. Aggregate vs. stratified F1 — the core comparison
3. Error taxonomy breakdown by drug class and model
4. Oncology deep dive: where and how each model fails
5. Few-shot recall degradation analysis
6. Downstream implications for RWE validity
7. Figures for blog post and presentation

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)

RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

---
## 1. Load and Validate Results

Each evaluation runner outputs a per-note results CSV. Schema per row: `note_id, gold_drug, gold_strength, drug_class, drug_found (0/1), strength_correct (0/1/None)`

**Expected files:**
- `results/regex_results.csv`
- `results/gpt4o_zero_shot_results.csv`
- `results/gpt4o_few_shot_results.csv`
- `results/biomistral_results.csv` (in progress)

In [ ]:
# Hardcoded results matching README table — replace with pd.read_csv() once CSVs are ready

RESULTS = {
    'Regex': {
        'drug_f1':     {'standard': 0.723, 'oncology': 0.141, 'prn': 0.648, 'overall': 0.739},
        'strength_f1': {'standard': 0.595, 'oncology': 0.000, 'prn': 0.489, 'overall': 0.626},
        'dataset': 'n2c2 2018'
    },
    'GPT-4o\n(zero-shot)': {
        'drug_f1':     {'standard': 0.737, 'oncology': 0.354, 'prn': 0.846, 'overall': 0.755},
        'strength_f1': {'standard': 0.847, 'oncology': 0.276, 'prn': 0.861, 'overall': 0.858},
        'dataset': 'MTSamples'
    },
    'GPT-4o\n(few-shot)': {
        'drug_f1':     {'standard': 0.592, 'oncology': 0.223, 'prn': 0.786, 'overall': 0.627},
        'strength_f1': {'standard': 0.808, 'oncology': 0.213, 'prn': 0.844, 'overall': 0.826},
        'dataset': 'MTSamples'
    },
}

CLASSES = ['standard', 'oncology', 'prn']
MODELS  = list(RESULTS.keys())
print('Models:', MODELS)
print('Drug classes:', CLASSES)

---
## 2. Aggregate vs. Stratified F1

This is the thesis figure. Aggregate Drug F1 spans 0.627-0.755 — narrow enough that a reader might conclude all three systems are broadly comparable. Stratified F1 collapses that conclusion.

The correct framing: **differential misclassification**. Systematic extraction failure in a specific drug class produces biased cohort composition, not merely noisy data. A study relying on extracted medication lists to identify chemotherapy-exposed patients will systematically undercount that cohort — and the undercounting will be correlated with documentation complexity, not random.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Aggregate F1 conceals systematic oncology failure', fontsize=13, fontweight='bold', y=1.01)

x      = np.arange(len(MODELS))
width  = 0.2
colors = {'standard': '#4C72B0', 'oncology': '#DD4949', 'prn': '#55A868', 'overall': '#888888'}

for ax_idx, metric in enumerate(['drug_f1', 'strength_f1']):
    ax = axes[ax_idx]
    for i, cls in enumerate(['standard', 'oncology', 'prn', 'overall']):
        vals = [RESULTS[m][metric][cls] for m in MODELS]
        ax.bar(x + (i - 1.5) * width, vals, width, label=cls.capitalize(),
               color=colors[cls], alpha=0.85 if cls != 'overall' else 0.45,
               edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS, fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('F1 Score')
    ax.set_title('Drug F1' if metric == 'drug_f1' else 'Strength F1')
    ax.axhline(0.5, color='black', linewidth=0.5, linestyle='--', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    if ax_idx == 1:
        ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_aggregate_vs_stratified_f1.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Error Taxonomy Breakdown

Aggregate failure rates tell you *how much* each model fails. Error taxonomy tells you *why* — and whether the failure mode is fixable with more data, better prompting, or is structural to the approach.

| Error type | Definition |
|---|---|
| `missing_drug` | Gold drug mention not found in output |
| `missing_dose` | Drug found, dose absent |
| `wrong_dose_value` | Numeric value incorrect |
| `wrong_dose_unit` | Value correct, unit wrong (e.g. mg vs mg/m²) |
| `hallucinated_drug` | Predicted drug has no gold counterpart |
| `hallucinated_dose` | Dose returned where gold has none |

The unit error (`wrong_dose_unit`) is particularly diagnostic for oncology: returning `75 mg` instead of `75 mg/m²` for paclitaxel means the model recognized the drug and retrieved a plausible value, but failed on the normalization convention distinguishing BSA-based from fixed dosing. This is not a recall problem — it is a representation problem.

In [ ]:
# TODO: load from results/errors.csv once evaluation reruns are complete
# error_df = pd.read_csv(RESULTS_DIR / 'errors.csv')
# error_summary = error_df.groupby(['model', 'drug_class', 'error_type']).size().reset_index(name='count')

ERROR_COLS = ['missing_drug', 'missing_dose', 'wrong_dose_value', 'wrong_dose_unit', 'hallucinated_drug', 'hallucinated_dose']
print('Error taxonomy analysis: pending results CSV')
print('Expected columns:', ERROR_COLS)

---
## 4. Oncology Deep Dive

Every model fails on oncology, but the failure modes differ — and that distinction matters for what you would do about it.

| Model | Primary failure mode | Structural reason |
|---|---|---|
| Regex + RxNorm | Drug recall (`missing_drug`) | Chemo agents underrepresented in RxNorm IN/BN terms; multi-word regimen names exceed 3-token window |
| GPT-4o zero-shot | Strength normalization (`wrong_dose_unit`) | AUC/BSA notation underrepresented in pretraining; model defaults to mg |
| GPT-4o few-shot | Drug recall (`missing_drug`) | Formatting bias toward structured lists; narrative chemo mentions missed |

**Convergent failure** across all three systems at the drug class level — despite different architectures and different failure mechanisms — is the strongest signal that the problem is data-structural, not model-specific.

In [ ]:
print('Drug F1 degradation: overall -> oncology')
print('-' * 50)
for model in MODELS:
    overall  = RESULTS[model]['drug_f1']['overall']
    oncology = RESULTS[model]['drug_f1']['oncology']
    label    = model.replace('\n', ' ')
    print(f"{label:<25} overall={overall:.3f}  oncology={oncology:.3f}  delta={oncology-overall:+.3f}")

print()
print('Strength F1 degradation: overall -> oncology')
print('-' * 50)
for model in MODELS:
    overall  = RESULTS[model]['strength_f1']['overall']
    oncology = RESULTS[model]['strength_f1']['oncology']
    label    = model.replace('\n', ' ')
    print(f"{label:<25} overall={overall:.3f}  oncology={oncology:.3f}  delta={oncology-overall:+.3f}")

---
## 5. Few-Shot Recall Degradation

The few-shot result is counterintuitive: prompting with examples *hurts* performance, specifically on recall. Precision stays high (~0.986). Recall drops from 0.610 to 0.460.

**Hypothesis:** The five in-context examples are drawn from structured medication list sections. Clinical notes also contain drug mentions in narrative prose — hospital course, assessment and plan, procedure notes. The formatting signal from the examples biases the model toward structured format, causing it to skip narrative drug mentions.

This is not a failure of few-shot prompting in general. It is a failure of **example selection strategy**. A well-designed prompt would deliberately include examples from narrative sections. Example diversity must match documentation diversity in the target corpus.

In [ ]:
# TODO: replace with actual P/R from results CSV
comparison = pd.DataFrame({
    'Model':   ['GPT-4o zero-shot', 'GPT-4o few-shot'],
    'Drug P':  [0.986, 0.986],
    'Drug R':  [0.610, 0.460],
    'Drug F1': [0.755, 0.627],
    'Str F1':  [0.858, 0.826],
})
print(comparison.to_string(index=False))
print()
print('Precision stable. Recall drops 0.15. This is formatting bias, not a knowledge gap.')

---
## 6. Downstream Implications for RWE Validity

This section translates extraction performance into the language of pharmacoepidemiology.

In epidemiology, non-random measurement error in an exposure variable produces **differential misclassification bias**. Systematic undercounting of chemotherapy agents in complex oncology notes means:

1. **Cohort contamination:** Studies defining chemotherapy-exposed cohorts will include unexposed patients.
2. **Selection bias:** Patients most likely to be misclassified are those with complex, narrative-heavy documentation — disproportionately patients with advanced disease and non-standard regimens. The extracted cohort is not a random sample of the true exposed population.
3. **Effect size distortion:** If the outcome of interest correlates with disease complexity, and disease complexity predicts extraction failure, effect estimates computed on the extracted cohort will be biased in an unpredictable direction.

None of this is visible from aggregate F1.

### Fitness-for-purpose threshold

Rather than asking "is F1 = 0.74 good enough?", ask what F1 threshold is required for the specific downstream use:

| Use case | Requirement |
|---|---|
| Cohort size estimation | Tolerates moderate recall loss if systematic |
| Comparative effectiveness research | High recall + low differential error across subgroups |
| Pharmacovigilance / drug safety signals | Near-perfect recall on the drug class under investigation |

By this framing, oncology Strength F1 of 0.000–0.276 is not just "low" — it is **disqualifying** for any dosing-dependent analysis of chemotherapy patients.

---
## 7. Figures

Saved to `results/figures/`. Planned outputs:
- `fig1_aggregate_vs_stratified_f1.png` — side-by-side bar chart (Section 2)
- `fig2_error_taxonomy_heatmap.png` — error type × drug class × model
- `fig3_oncology_delta.png` — overall vs. oncology F1 degradation per model
- `fig4_few_shot_precision_recall.png` — P/R decomposition, zero-shot vs. few-shot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

model_labels  = [m.replace('\n', ' ') for m in MODELS]
overall_drug  = [RESULTS[m]['drug_f1']['overall']  for m in MODELS]
oncology_drug = [RESULTS[m]['drug_f1']['oncology'] for m in MODELS]
x = np.arange(len(MODELS))

ax.bar(x - 0.2, overall_drug,  0.35, label='Overall Drug F1',  color='#4C72B0', alpha=0.75)
ax.bar(x + 0.2, oncology_drug, 0.35, label='Oncology Drug F1', color='#DD4949', alpha=0.85)

for i, (o, c) in enumerate(zip(overall_drug, oncology_drug)):
    ax.annotate('', xy=(x[i] + 0.2, c + 0.01), xytext=(x[i] - 0.2, o + 0.01),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Drug F1')
ax.set_title('Aggregate vs. Oncology Drug F1\n(arrows show within-model degradation)', fontsize=11)
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_oncology_delta.png', dpi=150, bbox_inches='tight')
plt.show()